# Student Performance Prediction Platform
### Machine Learning & Deep Learning Evaluation Pipeline
**SDG Alignment:** SDG 4 — Quality Education  
---
## Notebook Overview
This notebook documents the end-to-end Machine Learning and Deep Learning pipeline:
1. **Exploratory Data Analysis (EDA)** on 500 anonymized student continuous assessment records.
2. **Data Pre-processing & Class Imbalance Resolution** using SMOTE (Synthetic Minority Over-sampling Technique).
3. **Module 3 Unsupervised Learning:** K-Means Clustering with Elbow Method for student segmentation.
4. **Module 2 & 3 Supervised Learning:** Logistic Regression, Decision Tree, and Random Forest Ensemble.
5. **Module 4 Deep Learning:** Multi-Layer Perceptron (MLP) Neural Network with ReLU and Adam optimizer.
6. **Evaluation & Explainability:** Cross-validation, Confusion Matrices, Precision, Recall, and Feature Importance.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.neural_network import MLPClassifier
from sklearn.cluster import KMeans
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix, classification_report
from imblearn.over_sampling import SMOTE

sns.set_theme(style="whitegrid")
print("Libraries successfully imported!")

## 1. Load Dataset & Exploratory Data Analysis

In [ ]:
df = pd.read_csv("../data/students_dataset.csv")
print(f"Dataset shape: {df.shape}")
df.head()

In [ ]:
df.describe()

In [ ]:
print("Risk Level Distribution:")
print(df['risk_level'].value_counts(normalize=True) * 100)

## 2. Feature Pre-processing & SMOTE Balancing

In [ ]:
feature_cols = [
    "attendance", "internal_marks", "assignment_rate", "study_hours",
    "class_test_marks", "past_gpa", "parental_education",
    "extracurricular", "internet_access"
]
X = df[feature_cols].copy()
edu_map = {"High School": 0, "Bachelor": 1, "Master": 2}
X["parental_education"] = X["parental_education"].map(edu_map).fillna(1)

le = LabelEncoder()
le.classes_ = np.array(["High Risk", "Medium Risk", "Low Risk"])
y = le.transform(df["risk_level"])

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.20, random_state=42, stratify=y)
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# Apply SMOTE
smote = SMOTE(random_state=42)
X_train_smote, y_train_smote = smote.fit_resample(X_train_scaled, y_train)
print(f"Before SMOTE shape: {X_train.shape}, After SMOTE shape: {X_train_smote.shape}")

## 3. Module 3: Unsupervised K-Means Clustering & Elbow Method

In [ ]:
inertias = []
k_vals = range(2, 9)
for k in k_vals:
    km = KMeans(n_clusters=k, random_state=42, n_init=10)
    km.fit(scaler.transform(X))
    inertias.append(km.inertia_)

plt.figure(figsize=(7, 4))
plt.plot(k_vals, inertias, marker='o', color='#2563eb', linewidth=2)
plt.axvline(x=3, color='#dc2626', linestyle='--', label='Elbow Point (k=3)')
plt.title("K-Means Elbow Method")
plt.xlabel("k (Number of Clusters)")
plt.ylabel("Inertia")
plt.legend()
plt.show()

## 4. Supervised & Neural Network Model Training

In [ ]:
models = {
    "Logistic Regression": LogisticRegression(max_iter=1000, random_state=42),
    "Decision Tree": DecisionTreeClassifier(max_depth=6, min_samples_leaf=6, random_state=42),
    "Random Forest": RandomForestClassifier(n_estimators=120, max_depth=9, random_state=42),
    "MLP Neural Network": MLPClassifier(hidden_layer_sizes=(64, 32), activation='relu', solver='adam', alpha=0.005, max_iter=500, early_stopping=True, random_state=42)
}

for name, model in models.items():
    model.fit(X_train_smote, y_train_smote)
    preds = model.predict(X_test_scaled)
    acc = accuracy_score(y_test, preds)
    f1 = f1_score(y_test, preds, average='weighted')
    high_risk_rec = recall_score(y_test, preds, labels=[0], average=None)[0]
    print(f"{name:20s} | Acc: {acc*100:.2f}% | F1: {f1*100:.2f}% | High-Risk Recall: {high_risk_rec*100:.2f}%")

## 5. Feature Importance Analysis

In [ ]:
rf = models["Random Forest"]
feat_imp = pd.Series(rf.feature_importances_, index=feature_cols).sort_values(ascending=False)
plt.figure(figsize=(8, 4))
feat_imp.plot(kind='barh', color='#3b82f6')
plt.title("Random Forest Feature Importance")
plt.xlabel("Importance Score")
plt.gca().invert_yaxis()
plt.show()